# Merge MODIS Terra and Aqua Chlorophyll Data

This notebook merges MODIS Terra (morning overpass) and Aqua (afternoon overpass) chlorophyll data into single combined files for each lake.

## Overview
- **Input**: Separate Terra and Aqua CSV files for each lake
- **Output**: Combined CSV files with both Terra and Aqua observations
- **Processing**: Simple concatenation and sorting (no interpolation needed)
- **Lakes**: Detroit Lake and Upper Klamath Lake

## Key Points:
- Terra overpasses occur around 10:30 AM local time
- Aqua overpasses occur around 1:30 PM local time
- Each satellite provides unique daily observations (no overlap)
- Combined dataset provides up to 2 observations per day

In [ ]:
"""
Import required libraries for data processing.
"""

import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

# Set display options for better DataFrame viewing
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 10)

## Define Utility Functions

In [ ]:
def merge_modis_data(terra_file, aqua_file, output_file, lake_name):
    """
    Merge MODIS Terra and Aqua chlorophyll data for a single lake.
    
    Args:
        terra_file: Path to Terra CSV file
        aqua_file: Path to Aqua CSV file
        output_file: Path for merged output CSV
        lake_name: Name of lake for reporting
    
    Returns:
        DataFrame with merged data
    """
    print(f"\nProcessing {lake_name}...")
    print("="*60)
    
    # Read Terra data
    print(f"Reading Terra data from: {terra_file}")
    terra_df = pd.read_csv(terra_file)
    terra_df['date'] = pd.to_datetime(terra_df['date'])
    terra_df['satellite'] = 'Terra'
    print(f"  Terra records: {len(terra_df)}")
    print(f"  Date range: {terra_df['date'].min().date()} to {terra_df['date'].max().date()}")
    
    # Read Aqua data
    print(f"\nReading Aqua data from: {aqua_file}")
    aqua_df = pd.read_csv(aqua_file)
    aqua_df['date'] = pd.to_datetime(aqua_df['date'])
    aqua_df['satellite'] = 'Aqua'
    print(f"  Aqua records: {len(aqua_df)}")
    print(f"  Date range: {aqua_df['date'].min().date()} to {aqua_df['date'].max().date()}")
    
    # Combine datasets
    print("\nMerging datasets...")
    combined_df = pd.concat([terra_df, aqua_df], ignore_index=True)
    
    # Sort by date
    combined_df = combined_df.sort_values('date').reset_index(drop=True)
    
    # Add time of day indicator based on satellite
    combined_df['time_of_day'] = combined_df['satellite'].map({
        'Terra': 'morning',
        'Aqua': 'afternoon'
    })
    
    # Calculate basic statistics
    print("\nCombined dataset statistics:")
    print(f"  Total records: {len(combined_df)}")
    print(f"  Date range: {combined_df['date'].min().date()} to {combined_df['date'].max().date()}")
    print(f"  Terra observations: {len(combined_df[combined_df['satellite']=='Terra'])}")
    print(f"  Aqua observations: {len(combined_df[combined_df['satellite']=='Aqua'])}")
    
    # Check for same-day observations
    combined_df['date_only'] = combined_df['date'].dt.date
    same_day_counts = combined_df.groupby('date_only').size()
    days_with_both = (same_day_counts == 2).sum()
    print(f"  Days with both Terra and Aqua: {days_with_both}")
    
    # Chlorophyll statistics
    print(f"\nChlorophyll-a statistics (µg/L):")
    print(f"  Overall mean: {combined_df['chl'].mean():.2f}")
    print(f"  Overall median: {combined_df['chl'].median():.2f}")
    print(f"  Overall std: {combined_df['chl'].std():.2f}")
    print(f"  Min: {combined_df['chl'].min():.2f}")
    print(f"  Max: {combined_df['chl'].max():.2f}")
    print(f"  Terra mean: {combined_df[combined_df['satellite']=='Terra']['chl'].mean():.2f}")
    print(f"  Aqua mean: {combined_df[combined_df['satellite']=='Aqua']['chl'].mean():.2f}")
    
    # Save merged data
    print(f"\nSaving merged data to: {output_file}")
    # Drop the temporary date_only column before saving
    combined_df = combined_df.drop('date_only', axis=1)
    combined_df.to_csv(output_file, index=False)
    
    return combined_df

def plot_merged_data(df, lake_name):
    """
    Create visualization of merged MODIS data.
    
    Args:
        df: Merged dataframe
        lake_name: Name of lake for plot title
    """
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))
    
    # Time series plot with both satellites
    ax1 = axes[0]
    terra_data = df[df['satellite'] == 'Terra']
    aqua_data = df[df['satellite'] == 'Aqua']
    
    ax1.scatter(terra_data['date'], terra_data['chl'], 
                alpha=0.5, s=10, color='orange', label='Terra (AM)')
    ax1.scatter(aqua_data['date'], aqua_data['chl'], 
                alpha=0.5, s=10, color='blue', label='Aqua (PM)')
    
    ax1.set_xlabel('Date')
    ax1.set_ylabel('Chlorophyll-a (µg/L)')
    ax1.set_title(f'{lake_name} - MODIS Terra and Aqua Chlorophyll Time Series')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Histogram comparison
    ax2 = axes[1]
    bins = np.linspace(df['chl'].min(), df['chl'].quantile(0.95), 30)
    
    ax2.hist(terra_data['chl'], bins=bins, alpha=0.5, color='orange', 
             label=f'Terra (n={len(terra_data)})', edgecolor='black')
    ax2.hist(aqua_data['chl'], bins=bins, alpha=0.5, color='blue', 
             label=f'Aqua (n={len(aqua_data)})', edgecolor='black')
    
    ax2.set_xlabel('Chlorophyll-a (µg/L)')
    ax2.set_ylabel('Frequency')
    ax2.set_title(f'{lake_name} - Distribution of Chlorophyll Values by Satellite')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    # Save plot
    plot_filename = f"{lake_name.replace(' ', '_')}_MODIS_merged_analysis.png"
    fig.savefig(plot_filename, dpi=300, bbox_inches='tight')
    print(f"\nPlot saved as: {plot_filename}")
    
    plt.show()

## Process Detroit Lake MODIS Data

In [ ]:
"""
Merge MODIS Terra and Aqua data for Detroit Lake.
"""

# Define file paths
detroit_terra_file = 'Detroit_MODIS_Terra_500m_Chl_singlePixel.csv'
detroit_aqua_file = 'Detroit_MODIS_Aqua_500m_Chl_singlePixel.csv'
detroit_output_file = 'Detroit_MODIS_Combined_500m_Chl.csv'

# Merge data
detroit_combined = merge_modis_data(
    terra_file=detroit_terra_file,
    aqua_file=detroit_aqua_file,
    output_file=detroit_output_file,
    lake_name='Detroit Lake'
)

# Display first few rows
print("\nFirst 5 rows of merged data:")
print(detroit_combined.head())

# Display last few rows
print("\nLast 5 rows of merged data:")
print(detroit_combined.tail())

In [ ]:
"""
Visualize Detroit Lake merged MODIS data.
"""

plot_merged_data(detroit_combined, 'Detroit Lake')

## Process Upper Klamath Lake MODIS Data

In [ ]:
"""
Merge MODIS Terra and Aqua data for Upper Klamath Lake.

Note: The GEE script may have generated files with 'Klamath' prefix
instead of 'UpperKlamath'. Adjust filenames as needed.
"""

# Define file paths - check if files exist with different naming conventions
klamath_terra_file = 'Klamath_MODIS_Terra_500m_Chl_singlePixel.csv'
klamath_aqua_file = 'Klamath_MODIS_Aqua_500m_Chl_singlePixel.csv'
klamath_output_file = 'Klamath_MODIS_Combined_500m_Chl.csv'

# Check if files exist
from pathlib import Path

if not Path(klamath_terra_file).exists():
    print(f"Warning: {klamath_terra_file} not found.")
    print("Please check the filename or run the GEE extraction for Upper Klamath Lake.")
else:
    # Merge data
    klamath_combined = merge_modis_data(
        terra_file=klamath_terra_file,
        aqua_file=klamath_aqua_file,
        output_file=klamath_output_file,
        lake_name='Upper Klamath Lake'
    )
    
    # Display first few rows
    print("\nFirst 5 rows of merged data:")
    print(klamath_combined.head())
    
    # Display last few rows
    print("\nLast 5 rows of merged data:")
    print(klamath_combined.tail())

In [ ]:
"""
Visualize Upper Klamath Lake merged MODIS data.
"""

if 'klamath_combined' in locals():
    plot_merged_data(klamath_combined, 'Upper Klamath Lake')
else:
    print("Klamath data not available. Please check file paths.")

## Compare Lakes

In [ ]:
"""
Compare chlorophyll statistics between Detroit and Upper Klamath Lakes.
"""

if 'detroit_combined' in locals() and 'klamath_combined' in locals():
    # Create comparison table
    comparison = pd.DataFrame({
        'Metric': ['Mean Chl-a (µg/L)', 'Median Chl-a (µg/L)', 'Std Dev (µg/L)', 
                   'Min (µg/L)', 'Max (µg/L)', 'Total Observations',
                   'Terra Observations', 'Aqua Observations'],
        'Detroit Lake': [
            detroit_combined['chl'].mean(),
            detroit_combined['chl'].median(),
            detroit_combined['chl'].std(),
            detroit_combined['chl'].min(),
            detroit_combined['chl'].max(),
            len(detroit_combined),
            len(detroit_combined[detroit_combined['satellite']=='Terra']),
            len(detroit_combined[detroit_combined['satellite']=='Aqua'])
        ],
        'Upper Klamath Lake': [
            klamath_combined['chl'].mean(),
            klamath_combined['chl'].median(),
            klamath_combined['chl'].std(),
            klamath_combined['chl'].min(),
            klamath_combined['chl'].max(),
            len(klamath_combined),
            len(klamath_combined[klamath_combined['satellite']=='Terra']),
            len(klamath_combined[klamath_combined['satellite']=='Aqua'])
        ]
    })
    
    # Format numeric columns
    for col in ['Detroit Lake', 'Upper Klamath Lake']:
        comparison.loc[:4, col] = comparison.loc[:4, col].round(2)
        comparison.loc[5:, col] = comparison.loc[5:, col].astype(int)
    
    print("\n" + "="*60)
    print("COMPARISON BETWEEN LAKES")
    print("="*60)
    print(comparison.to_string(index=False))
    
    # Create side-by-side box plots
    fig, ax = plt.subplots(1, 1, figsize=(10, 6))
    
    detroit_data = [detroit_combined[detroit_combined['satellite']=='Terra']['chl'].values,
                    detroit_combined[detroit_combined['satellite']=='Aqua']['chl'].values]
    klamath_data = [klamath_combined[klamath_combined['satellite']=='Terra']['chl'].values,
                    klamath_combined[klamath_combined['satellite']=='Aqua']['chl'].values]
    
    positions = [1, 2, 4, 5]
    colors = ['orange', 'blue', 'orange', 'blue']
    
    bp = ax.boxplot(detroit_data + klamath_data, positions=positions, widths=0.6,
                    patch_artist=True, showfliers=False)
    
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.5)
    
    ax.set_xticks([1.5, 4.5])
    ax.set_xticklabels(['Detroit Lake', 'Upper Klamath Lake'])
    ax.set_ylabel('Chlorophyll-a (µg/L)')
    ax.set_title('MODIS Chlorophyll Comparison by Lake and Satellite')
    ax.grid(True, alpha=0.3)
    
    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor='orange', alpha=0.5, label='Terra (AM)'),
                      Patch(facecolor='blue', alpha=0.5, label='Aqua (PM)')]
    ax.legend(handles=legend_elements)
    
    plt.tight_layout()
    plt.savefig('Lakes_MODIS_Comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    
else:
    print("Both lake datasets needed for comparison. Please check file availability.")

## Summary

This notebook has successfully merged MODIS Terra and Aqua chlorophyll data for both Detroit Lake and Upper Klamath Lake. The merged files contain:

1. **Date**: Observation date
2. **Chlorophyll**: Chlorophyll-a concentration in µg/L
3. **Sensor**: Original sensor tag from GEE processing
4. **Satellite**: Terra or Aqua identifier
5. **Time of day**: Morning (Terra) or Afternoon (Aqua)

The combined datasets provide enhanced temporal resolution for monitoring algae dynamics in both lakes.